In [1]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

:: loading settings :: url = jar:file:/home/coder/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
ru.yandex.clickhouse#clickhouse-jdbc added as a dependency
org.postgresql#postgresql added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7606a98d-d282-4d05-8a54-9f4cebbb92c6;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found ru.yandex.clickhouse#clickhouse-jdbc;0.3.2 in central
	found com.clickhouse#clickhouse-http-client;0.3.2 in central
	found com.clickhouse#clickho

## Загрузка данных из postgresSql

In [2]:
import os
# ⬇️ Параметры подключения к PostgreSQL 
jdbc_url = "jdbc:postgresql://postgres_source:5432/source"
table_name_shop = "public.shops"
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")


df_shop = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("user", db_user) \
    .option("password", db_password) \
    .option("dbtable", table_name_shop) \
    .option("fetchsize", 1000) \
    .option("driver", "org.postgresql.Driver") \
    .load()


df_shop.show()


table_name_shop_timezone = "public.shop_timezone"
df_shop_timezone = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("user", db_user) \
    .option("password", db_password) \
    .option("dbtable", table_name_shop_timezone) \
    .option("fetchsize", 1000) \
    .option("driver", "org.postgresql.Driver") \
    .load() 

df_shop_timezone.show()

+-----+-----------+
|st_id|  shop_name|
+-----+-----------+
|  842|      Lenta|
|  843|     Magnit|
|  844|       Spar|
|  845|Pyaterochka|
|  846|      Lenta|
|  847|      Diksi|
|  848|      Lenta|
|  849|   FixPrice|
|  850|     Magnit|
|  851|      Lenta|
+-----+-----------+

+-----+---------+
|plant|time_zone|
+-----+---------+
|  842|         |
|  842|    RUS07|
|  843|    RUS04|
|  844|         |
|  845|         |
|  845|    RUS05|
|  847|    RUS03|
|  848|    RUS08|
|  848|         |
| P847|         |
| E103|    RUS08|
| -134|    RUS04|
|    0|         |
|    0|    RUS08|
|  848|         |
+-----+---------+



## Обработка данных таблицы по тз

In [14]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

df_shop_timezone_transform = (
    df_shop_timezone
    .where(F.col("plant").rlike(r"^\d+$")) #числовые  id
    .where(F.col("time_zone").isNotNull() & (F.col("time_zone") != ""))
    .select(F.col("plant").cast("int").alias("id"), "time_zone")
)


df_shop_timezone_transform.show()

+---+---------+
| id|time_zone|
+---+---------+
|842|    RUS07|
|843|    RUS04|
|845|    RUS05|
|847|    RUS03|
|848|    RUS08|
|  0|    RUS08|
+---+---------+



In [16]:
w_by_shop = Window.partitionBy("shop_name")

final_df = (
    df_shop
    .join(
        df_shop_timezone_transform,
        df_shop.st_id == df_shop_timezone_transform.id,
        "left"
    )
    .withColumn("tz_code_raw", F.substring(F.col("time_zone"), -1, 1))
    .withColumn("tz_code_shop", F.first("tz_code_raw", ignorenulls=True).over(w_by_shop))
    .withColumn("tz_code", F.coalesce(F.col("tz_code_raw"), F.col("tz_code_shop"), F.lit(3)))
    .select("st_id", "shop_name", "tz_code")
    .orderBy(F.col("st_id"))
)

final_df.show()

+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      7|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      5|
|  846|      Lenta|      7|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
|  849|   FixPrice|      3|
|  850|     Magnit|      4|
|  851|      Lenta|      7|
+-----+-----------+-------+

